In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Clean CSV not found at: {csv_path}")
    return pd.read_csv(csv_path).reset_index(drop=True)


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32) -> torch.Tensor:
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, device, dataset_names):
    print(f"Using device: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)
    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")
        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)
        all_data[name] = {"df": df, "emb": embeddings, "labels": labels,
                           "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask}
    return all_data


class TextOnlyClassifier(nn.Module):
    def __init__(self, emb_dim, hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden)
        self.fc2 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = self.dropout(h)
        return self.fc2(h)


def resample_train_mask(labels, orig_train_mask, target_fake_ratio, seed):
    """
    NOTE on naming: this targets the label==1 (fake) class share directly, matching
    the corrected label convention (0=real, 1=fake) confirmed from prepare_datasets.py.
    """
    rng = np.random.RandomState(seed)
    train_idx = torch.nonzero(orig_train_mask, as_tuple=True)[0].numpy()
    train_labels = labels[orig_train_mask].numpy()
    fake_idx = train_idx[train_labels == 1]
    real_idx = train_idx[train_labels == 0]
    Fk, R = len(fake_idx), len(real_idx)

    p = target_fake_ratio  # fraction that should be label==1 (fake)
    new_fake_a = Fk
    new_real_a = round(Fk * (1 - p) / p) if p > 0 else R
    new_real_b = R
    new_fake_b = round(R * p / (1 - p)) if p < 1 else Fk

    if new_real_a <= R:
        new_fake, new_real = new_fake_a, new_real_a
    else:
        new_fake, new_real = new_fake_b, new_real_b

    new_fake = max(1, min(new_fake, Fk))
    new_real = max(1, min(new_real, R))

    chosen_fake = rng.choice(fake_idx, size=new_fake, replace=False)
    chosen_real = rng.choice(real_idx, size=new_real, replace=False)
    chosen = np.concatenate([chosen_fake, chosen_real])

    new_mask = torch.zeros_like(orig_train_mask)
    new_mask[chosen] = True
    actual_ratio = new_fake / (new_fake + new_real)
    return new_mask, actual_ratio


def lr_lambda(epoch, total_epochs, warmup_frac):
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


def train_one_run(
    src_name, all_data, device, target_fake_ratio, seed, warmup_frac,
    epochs=30, lr=1e-3, weight_decay=1e-4, hidden=256, dropout=0.3,
    grad_clip_norm=1.0, collapse_epoch_start=1, collapse_threshold=0.98,
):
    """
    UNWEIGHTED loss throughout (we already know weighted never collapses -- this
    experiment is specifically about whether warmup length controls the collapse
    ONSET epoch in the unweighted, collapse-prone regime).
    """
    set_seed(seed)

    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    orig_train_mask = data_src["train_mask"]
    val_mask = data_src["val_mask"].to(device)

    resampled_mask, actual_ratio = resample_train_mask(
        data_src["labels"], orig_train_mask, target_fake_ratio, seed
    )
    train_mask = resampled_mask.to(device)

    emb_dim = x_src.size(1)
    model = TextOnlyClassifier(emb_dim=emb_dim, hidden=hidden, num_classes=2, dropout=dropout).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs, warmup_frac)
    )
    criterion = nn.CrossEntropyLoss()  # unweighted, deliberately

    expected_warmup_end_epoch = max(1, int(epochs * warmup_frac)) - 1  # 0-indexed epoch where warmup completes

    epoch_first_collapse = None
    val_pred_fake_frac_curve: List[float] = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x_src)
        loss = criterion(logits[train_mask], y_src[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits_full = model(x_src)
            val_pred = val_logits_full[val_mask].argmax(dim=-1)
            pred_fake_frac = val_pred.float().mean().item()  # label 1 = fake

        val_pred_fake_frac_curve.append(pred_fake_frac)
        is_collapsed_now = (pred_fake_frac >= collapse_threshold) or (pred_fake_frac <= (1 - collapse_threshold))
        if is_collapsed_now and epoch >= collapse_epoch_start and epoch_first_collapse is None:
            epoch_first_collapse = epoch

    final_pred_fake_frac = val_pred_fake_frac_curve[-1]
    collapsed_final = (final_pred_fake_frac >= collapse_threshold) or (final_pred_fake_frac <= (1 - collapse_threshold))

    return {
        "source": src_name, "warmup_frac": warmup_frac,
        "expected_warmup_end_epoch": expected_warmup_end_epoch,
        "seed": seed, "target_fake_ratio": target_fake_ratio, "actual_fake_ratio": actual_ratio,
        "epoch_first_collapse": epoch_first_collapse if epoch_first_collapse is not None else -1,
        "collapsed_final": bool(collapsed_final),
    }


def run_warmup_confirmation(
    base_dir, sources=["coaid", "fakenewsnet"], target_fake_ratio=0.80,
    warmup_fracs=[0.1, 0.2, 0.3, 0.5, 0.7], seeds=[0, 1, 2],
    model_name="roberta-base", epochs=30, max_length=128, batch_size=32, dropout=0.3,
):
    device = get_device()
    all_data = prepare_all_datasets(
        base_dir=base_dir, model_name=model_name, max_length=max_length,
        batch_size=batch_size, device=device, dataset_names=sources,
    )

    out_dir = os.path.join(base_dir, "warmup_confirmation_results")
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    total_runs = len(sources) * len(warmup_fracs) * len(seeds)
    counter = 0

    print(f"\n=== Starting warmup-length confirmation: {total_runs} runs "
          f"(fixed fake-class train ratio={target_fake_ratio}, unweighted loss throughout) ===")
    for src_name in sources:
        for warmup_frac in warmup_fracs:
            for seed in seeds:
                counter += 1
                print(f"\n[{counter}/{total_runs}] source={src_name} warmup_frac={warmup_frac} seed={seed}")
                result = train_one_run(
                    src_name=src_name, all_data=all_data, device=device,
                    target_fake_ratio=target_fake_ratio, seed=seed, warmup_frac=warmup_frac,
                    epochs=epochs, dropout=dropout,
                )
                print(
                    f"  expected_warmup_end_epoch={result['expected_warmup_end_epoch']} "
                    f"epoch_first_collapse={result['epoch_first_collapse']} "
                    f"collapsed_final={result['collapsed_final']}"
                )
                rows.append(result)

    df = pd.DataFrame(rows)
    out_path = os.path.join(out_dir, "warmup_confirmation_summary.csv")
    df.to_csv(out_path, index=False)
    print(f"\n=== Warmup confirmation complete. Saved to: {out_path} ===")

    print("\n=== Mean epoch_first_collapse by warmup_frac (only collapsed runs, -1 excluded) ===")
    collapsed_only = df[df["epoch_first_collapse"] >= 0]
    summary = collapsed_only.groupby(["source", "warmup_frac"]).agg(
        expected_warmup_end_epoch=("expected_warmup_end_epoch", "first"),
        mean_first_collapse_epoch=("epoch_first_collapse", "mean"),
        n_collapsed=("epoch_first_collapse", "count"),
    ).reset_index()
    print(summary.to_string(index=False))

    print(f"\nTotal collapse rate: {df['collapsed_final'].mean()*100:.1f}% "
          f"({df['collapsed_final'].sum()}/{len(df)})")

    return df


run_warmup_confirmation(
    base_dir="drive/MyDrive/CSML/data",
    sources=["coaid", "fakenewsnet"],
    target_fake_ratio=0.80,
    warmup_fracs=[0.1, 0.2, 0.3, 0.5, 0.7],
    seeds=[0, 1, 2],
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    batch_size=32,
    dropout=0.3,
)
